# IndexTTS2 — Zero-Shot TTS with Emotion Control

This notebook runs IndexTTS2 on Google Colab (Python 3.12 + CUDA).

**Steps:**
1. Install dependencies from the private repo
2. Download model checkpoints
3. Run inference (CLI or WebUI)

## 0. Check GPU

In [ ]:
!nvidia-smi
import sys
print(f"Python {sys.version}")

## 1. Clone repo & install dependencies

Colab's notebook kernel runs on the system Python, so we install directly with `pip`.

In [ ]:
# Clone the repo (py3.12 branch)
%cd /content
!git clone -b py3.12 https://github.com/deluxebear/index-tts.git
%cd /content/index-tts

In [ ]:
# Step 1: Uninstall Colab's pre-installed packages that conflict with project deps
!pip uninstall -y torch torchvision torchaudio tensorflow keras tensorboard protobuf 2>/dev/null

# Step 2: Install project + matching torch 2.8 from the CUDA 12.8 index, plus ninja for CUDA kernels
!pip install ninja
!pip install -e ".[webui]" --extra-index-url https://download.pytorch.org/whl/cu128

# Step 3: Verify
import torch
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")
import transformers, numba, librosa
print(f"transformers={transformers.__version__}, numba={numba.__version__}, librosa={librosa.__version__}")

## 2. Download model checkpoints

In [ ]:
# Download IndexTTS-2 checkpoints from HuggingFace
!huggingface-cli download IndexTeam/IndexTTS-2 --local-dir checkpoints

## 3. Upload voice prompt

IndexTTS2 is a zero-shot voice cloning model — it needs a reference audio to determine the voice style. Upload a WAV file (3~10 seconds recommended).

In [ ]:
from google.colab import files

print("Upload a WAV file as your voice prompt (3~10 seconds recommended):")
uploaded = files.upload()
prompt_wav = list(uploaded.keys())[0]
print(f"Using: {prompt_wav}")

## 4. Load model & run inference

In [ ]:
from indextts.infer_v2 import IndexTTS2

tts = IndexTTS2(
    cfg_path="checkpoints/config.yaml",
    model_dir="checkpoints",
    use_fp16=True,
)

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)
from IPython.display import Audio, display

# Chinese text
text = "大家好，我现在正在体验AI语音合成技术，效果真的非常惊艳！"
output_path = "outputs/test_zh.wav"
tts.infer(spk_audio_prompt=prompt_wav, text=text, output_path=output_path, verbose=True)
display(Audio(output_path))

## 5. Emotion control

In [ ]:
# Emotion vector: [happy, angry, sad, afraid, disgusted, melancholic, surprised, calm]
text = "今天天气真好，心情特别愉快！"

# Happy emotion
emo_vec_happy = [1.0, 0, 0, 0, 0, 0, 0, 0]
emo_vec_happy = tts.normalize_emo_vec(emo_vec_happy, apply_bias=True)
tts.infer(spk_audio_prompt=prompt_wav, text=text, output_path="outputs/test_happy.wav",
          emo_vector=emo_vec_happy, verbose=True)
print("--- Happy ---")
display(Audio("outputs/test_happy.wav"))

# Sad emotion
emo_vec_sad = [0, 0, 1.0, 0, 0, 0, 0, 0]
emo_vec_sad = tts.normalize_emo_vec(emo_vec_sad, apply_bias=True)
tts.infer(spk_audio_prompt=prompt_wav, text=text, output_path="outputs/test_sad.wav",
          emo_vector=emo_vec_sad, verbose=True)
print("--- Sad ---")
display(Audio("outputs/test_sad.wav"))

## 6. Launch WebUI (optional)

This starts a Gradio WebUI with a public share link.

In [ ]:
# Launch WebUI with a public share link
!python webui.py --fp16 --port 7860